# Development and testing for IPP post-tariff population

## Script test

In [52]:
%load_ext autoreload
%autoreload 2

In [60]:
from src.visualization.sentencing import plotly_ipp_post_tariff

## Script development

In [1]:
# Importing libraries
import os

import chart_studio
import chart_studio.plotly as py
import pandas as pd
import plotly.graph_objs as go
import plotly.io as pio
from dotenv import find_dotenv, load_dotenv
from plotly.subplots import make_subplots

# Import prt_theme module
from src.visualization import prt_theme

# Loading environment variables
dotenv_path = find_dotenv()
load_dotenv(dotenv_path)

# Adding plotly credentials
chart_studio.tools.set_credentials_file(
    username=os.getenv("PLOTLY_USERNAME"), api_key=os.getenv("PLOTLY_API_KEY")
)

#Setting default Plotly template and assigning attributes to prt_template
pio.templates.default = "prt_template"
prt_template = prt_theme.pio.templates['prt_template']

In [29]:
df = (
    pd.read_csv("data/processed/sentencing/ipp_post_tariff.csv", usecols=['Years over tariff', 'short_year', 'number'], dtype={'number': 'Int64'})
    .dropna()
    .rename(columns={'Years over tariff': 'years', 'short_year': 'short_years', 'number': 'value'})
    # .replace(to_replace="<1", value="Less than 1")
    )

df

,years,short_years,value
0,Less than 1 year,<1,16
1,From 1 year to less than 2 years,1,13
2,From 2 years to less than 3 years,2,17
3,From 3 years to less than 4 years,3,28
4,From 4 years to less than 5 years,4,50
5,From 5 years to less than 6 years,5,53
6,From 6 years to less than 7 years,6,70
7,From 7 years to less than 8 years,7,99
8,From 8 years to less than 9 years,8,85
9,From 9 years to less than 10 years,9,132


In [41]:
# Calculate the cumulative frequency
df["cumulative_sum"] = df["value"].cumsum()

# Calculate the total frequency
total = df["value"].sum()

# Find the median
median_year = int(df.loc[df["cumulative_sum"] >= total / 2, "short_years"].iloc[0])
print(f"The median year is: {median_year}")

The median year is: 10


In [42]:
len(df['short_years'])

19

In [45]:
type(median_year)

int

In [47]:
marker_color = [prt_template.layout.colorway[0],] * len(df['short_years'])
marker_color[median_year] = prt_template.layout.colorway[1]

In [51]:
## Plotting
fig = go.Figure()

fig.add_traces(
    go.Bar(
        x=df["short_years"],
        y=df["value"],
        customdata=df['years'],
        hovertemplate="%{y} people<extra>%{customdata}</extra>",
        marker_color=marker_color,
    )
)

fig.update_layout(
    hovermode='x',
    yaxis_tickformat= ",.0f",
    yaxis_automargin=True, #To avoid clipping of y-axis labels
    xaxis_dtick=1,
    margin_pad=5,
    margin=dict(t=20, b=25, l=0, r=25),
)

fig.update_yaxes(
    range=[0,145],
)

fig.update_xaxes(
    type='category',
)

## Chart annotations
annotations = []

# Add y-axis label annotation with placement based on dataframe column
prt_theme.add_annotation(
    annotations, "People in prison", annotation_type="y-axis"
)

# Adding annotations to layout
fig.update_layout(annotations=annotations)
fig.show()

In [21]:
fig.data[0]

Bar({
    'hovertemplate': '%{y}',
    'x': array(['< 1', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12',
                '13', '14', '15', '16', '17', '18'], dtype=object),
    'y': array([ 16,  13,  17,  28,  50,  53,  70,  99,  85, 132, 132, 117, 128, 128,
                 94,  62,  21,   1,   1])
})